In [17]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import datetime

DIRS = {
    'models': '../models',
    'ltr_data': '../data/processed/ltr_datasets'
}

# Load tuned model with fallback
tuned_path = f"{DIRS['models']}/xgboost_tuned.pkl"
baseline_path = f"{DIRS['models']}/xgboost_classifier.pkl"

if os.path.exists(tuned_path):
    xgb_model = joblib.load(tuned_path)
    model_label = "TUNED"
    print(f"Loaded TUNED model")
else:
    xgb_model = joblib.load(baseline_path)
    model_label = "BASELINE"
    print(f"Loaded BASELINE model")

# Get expected feature columns
model_feature_columns = xgb_model.get_booster().feature_names
if not model_feature_columns:
    with open(f"{DIRS['models']}/feature_columns.json", 'r') as f:
        meta = json.load(f)
    model_feature_columns = meta.get('features', meta) if isinstance(meta, dict) else meta

print(f"Model: XGBoost Classifier ({model_label})")
print(f"Expected features: {len(model_feature_columns)}")

# Load candidate pool and training data
candidate_pool = pd.read_csv(f"{DIRS['ltr_data']}/employee_candidate_pool.csv")
if "Employee_Name" not in candidate_pool.columns:
    candidate_pool["Employee_Name"] = candidate_pool["Employee_ID"].astype(str)

ltr_train = pd.read_csv(f"{DIRS['ltr_data']}/ltr_train_dataset.csv")
print(f"Loaded {len(ltr_train)} training rows, {len(candidate_pool)} candidates")


Loaded TUNED model
Model: XGBoost Classifier (TUNED)
Expected features: 56
Loaded 36605 training rows, 49 candidates


In [18]:
# Build real employee feature profiles from training data
relevant = ltr_train[ltr_train['relevance'] == 1].copy()

# Aggregate historical features
agg_dict = {'Task_ID': 'count'}
if 'Estimated_Planned_Hours' in relevant.columns:
    agg_dict['Estimated_Planned_Hours'] = 'mean'
if 'Task_Skill_Count' in relevant.columns:
    agg_dict['Task_Skill_Count'] = 'mean'
if 'Task_Text_Length' in relevant.columns:
    agg_dict['Task_Text_Length'] = 'mean'
if 'Days_To_Deadline' in relevant.columns:
    agg_dict['Days_To_Deadline'] = 'mean'

emp_features = relevant.groupby('Employee_ID').agg(agg_dict).reset_index()
emp_features = emp_features.rename(columns={
    'Task_ID': 'employee_historical_task_count',
    'Estimated_Planned_Hours': 'employee_historical_avg_planned_hours',
    'Task_Skill_Count': 'employee_historical_avg_skill_count',
    'Task_Text_Length': 'employee_historical_avg_task_text_length',
    'Days_To_Deadline': 'employee_historical_avg_deadline_days'
})

# Unique projects
if 'Project_Name' in relevant.columns:
    proj_unique = relevant.groupby('Employee_ID')['Project_Name'].nunique().reset_index()
    proj_unique.columns = ['Employee_ID', 'employee_historical_unique_projects']
    emp_features = emp_features.merge(proj_unique, on='Employee_ID', how='left')
else:
    emp_features['employee_historical_unique_projects'] = 1

emp_features['employee_historical_project_count'] = emp_features['employee_historical_unique_projects']
emp_features['employee_log_task_count'] = np.log1p(emp_features['employee_historical_task_count'])

# Employee skill profiles (average skill presence in their tasks)
skill_cols = [c for c in relevant.columns if c.startswith('Skill_')]
if skill_cols:
    emp_skill = relevant.groupby('Employee_ID')[skill_cols].mean().reset_index()
    emp_skill = emp_skill.rename(columns={c: c.replace('Skill_', 'Employee_Profile_Skill_') for c in skill_cols})
    emp_features = emp_features.merge(emp_skill, on='Employee_ID', how='left')

print(f"Built profiles for {len(emp_features)} employees")
print(f"Top 5 by task count:")
print(emp_features.nlargest(5, 'employee_historical_task_count')[['Employee_ID', 'employee_historical_task_count']].to_string(index=False))


Built profiles for 49 employees
Top 5 by task count:
Employee_ID  employee_historical_task_count
     EMP-11                             178
     EMP-16                             161
      EMP-5                             151
      EMP-8                             128
      EMP-7                             124


In [19]:
# Comprehensive skill keyword dictionary
# Based on the skill taxonomy from notebook 04 (Feature Engineering)
SKILL_KEYWORDS = {
    'Skill_Odoo_ERP_Development': [
        'odoo', 'erp', 'addon', 'addons', 'module', 'modules', 'sh maintain',
        'odoo.sh', 'openerp', 'github', 'repository', 'build'
    ],
    'Skill_Database_Management': [
        'database', 'sql', 'db backup', 'data import', 'restore', 'db ',
        'table', 'query', 'postgres', 'mysql', 'backup'
    ],
    'Skill_Server_Administration': [
        'server', 'vps', 'ssl', 'deployment', 'deploy', 'pipeline', 'ci/cd',
        'docker', 'aws', 'github actions', 'jenkins', 'infrastructure', 'hosting',
        'domain', 'configuration'
    ],
    'Skill_Project_Management': [
        'meeting', 'plan', 'scrum', 'planning', 'sprint', 'discussion',
        'requirement', 'requirement gathering', 'kickoff', 'standup'
    ],
    'Skill_Software_Testing': [
        'test', 'testing', 'uat', 'qa', 'bug', 'quality', 'automation',
        'test case', 'test plan', 'defect', 'verification'
    ],
    'Skill_Web_Development': [
        'web', 'frontend', 'backend', 'website', 'html', 'css', 'javascript',
        'api', 'microservice', 'rest', 'endpoint', 'integration'
    ],
    'Skill_Client_and_Functional_Support': [
        'client', 'support', 'functional', 'live support', 'troubleshooting',
        'customer', 'end user', 'helpdesk', 'functional support'
    ],
    'Skill_Documentation': [
        'document', 'documentation', 'report', 'blueprint', 'preparing report',
        'write', 'manual', 'guide', 'sop', 'spec'
    ],
    'Skill_Training_and_Mentorship': [
        'training', 'train', 'mentor', 'mentorship', 'teach', 'onboard',
        'workshop', 'tutorial', 'demo', 'walkthrough', 'explain'
    ]
}

def detect_task_skills(task_text):
    """Return dict of {skill_col: 0/1} based on keyword matching."""
    text_lower = str(task_text).lower()
    skills = {}
    for skill_col, keywords in SKILL_KEYWORDS.items():
        skills[skill_col] = int(any(kw in text_lower for kw in keywords))
    return skills

# Test with the demo task
test_desc = "Set up automated testing and deployment pipelines using GitHub Actions for the inventory and order management microservices."
test_skills = detect_task_skills(test_desc)
print("Skill detection test for demo task:")
for skill, detected in test_skills.items():
    if detected:
        print(f"  ✓ {skill}")
print(f"  Total: {sum(test_skills.values())} skills detected")


Skill detection test for demo task:
  ✓ Skill_Odoo_ERP_Development
  ✓ Skill_Server_Administration
  ✓ Skill_Software_Testing
  ✓ Skill_Web_Development
  Total: 4 skills detected


In [20]:
SKILL_COLS = [c for c in model_feature_columns if c.startswith('Skill_') and not c.startswith('Employee_Profile_')]

def extract_task_features(task_df, candidates_df, emp_features_df):
    """
    Build feature matrix for inference: one row per (task, candidate).
    Uses REAL employee historical features from training data.
    """
    out = candidates_df[['Employee_ID', 'Employee_Name']].copy()
    
    # Task-level features (same for all candidates)
    task_desc = str(task_df['Task_Description'].iloc[0]) if 'Task_Description' in task_df.columns else ''
    task_title = str(task_df['Task_Title'].iloc[0]) if 'Task_Title' in task_df.columns else ''
    full_text = task_desc + ' ' + task_title
    
    out['Task_Text_Length'] = len(task_desc)
    out['Task_Word_Count'] = len(task_desc.split())
    out['Task_Description_Length'] = len(task_desc)
    out['Task_Name_Length'] = len(task_title)
    out['Has_Task_Description'] = 1 if task_desc else 0
    
    hours = float(task_df['Estimated_Planned_Hours'].iloc[0]) if 'Estimated_Planned_Hours' in task_df.columns else 0.0
    out['Estimated_Planned_Hours'] = hours
    out['Planned_Hours_Log'] = np.log1p(hours)
    out['Planned_Task_Size_Small'] = 1 if hours <= 8 else 0
    out['Planned_Task_Size_Medium'] = 1 if 8 < hours <= 40 else 0
    out['Planned_Task_Size_Large'] = 1 if hours > 40 else 0
    
    if 'Days_To_Deadline' in task_df.columns:
        out['Days_To_Deadline'] = float(task_df['Days_To_Deadline'].iloc[0])
        out['Has_Deadline'] = 1
    else:
        out['Days_To_Deadline'] = np.nan
        out['Has_Deadline'] = 0
    
    # Skill detection from task text
    task_skills = detect_task_skills(full_text)
    for skill_col, val in task_skills.items():
        out[skill_col] = val
    out['Task_Skill_Count'] = sum(task_skills.values())
    
    # Date features (today as proxy for created date)
    today = datetime.datetime.now()
    out['Created_Year'] = today.year
    out['Created_Month'] = today.month
    out['Created_DayOfWeek'] = today.weekday()
    out['Created_Quarter'] = (today.month - 1) // 3 + 1
    
    # Priority
    priority = task_df['Task_Priority'].iloc[0] if 'Task_Priority' in task_df.columns else 'Low'
    out['Task_Priority_Low'] = 1 if priority == 'Low' else 0
    out['Task_Priority_Normal'] = 1 if priority != 'Low' else 0
    
    # Employee features lookup
    emp_lookup = emp_features_df.set_index('Employee_ID')
    
    # Map core employee features
    out['employee_historical_task_count'] = out['Employee_ID'].map(
        emp_lookup['employee_historical_task_count']
    ).fillna(emp_features_df['employee_historical_task_count'].median())
    
    out['employee_historical_unique_projects'] = out['Employee_ID'].map(
        emp_lookup['employee_historical_unique_projects']
    ).fillna(emp_features_df['employee_historical_unique_projects'].median())
    
    # Map other employee features
    for col in emp_features_df.columns:
        if col in ['Employee_ID', 'employee_historical_task_count', 'employee_historical_unique_projects']:
            continue
        if col in model_feature_columns:
            default = emp_features_df[col].median() if pd.api.types.is_numeric_dtype(emp_features_df[col]) else 0
            out[col] = out['Employee_ID'].map(emp_lookup[col]).fillna(default)
    
    # Project experience (average per project)
    out['employee_project_task_count'] = (
        out['employee_historical_task_count'] / out['employee_historical_unique_projects'].clip(lower=1)
    )
    out['employee_has_project_experience'] = (out['employee_project_task_count'] > 0).astype(int)
    
    # Skill match score
    if SKILL_COLS and out['Task_Skill_Count'].max() > 0:
        emp_skill_match = pd.Series(0.0, index=out.index)
        for skill_col in SKILL_COLS:
            emp_profile_col = skill_col.replace('Skill_', 'Employee_Profile_Skill_')
            if emp_profile_col in out.columns:
                emp_skill_match = emp_skill_match + (out[skill_col].astype(float) * out[emp_profile_col].astype(float))
        out['employee_task_skill_match_count'] = emp_skill_match.astype(int)
        out['employee_task_skill_match_ratio'] = emp_skill_match / out['Task_Skill_Count']
        out['employee_has_matching_skill'] = (out['employee_task_skill_match_count'] > 0).astype(int)
    else:
        out['employee_task_skill_match_count'] = 0
        out['employee_task_skill_match_ratio'] = 0.0
        out['employee_has_matching_skill'] = 0
    
    # Fill missing features with NaN (XGBoost handles natively)
    for col in model_feature_columns:
        if col not in out.columns:
            out[col] = np.nan
    
    return out


In [21]:
def recommend_employees(raw_task_dict, candidate_df, model, emp_features_df, expected_features, top_k=5):
    """Generate ranked employee recommendations for a new task."""
    print(f"\nAnalyzing Task: {raw_task_dict.get('Task_Title', 'Unknown')}...")
    
    # Show detected skills
    desc = raw_task_dict.get('Task_Description', '') + ' ' + raw_task_dict.get('Task_Title', '')
    detected = detect_task_skills(desc)
    detected_skills = [s.replace('Skill_', '') for s, v in detected.items() if v]
    print(f"Detected skills: {', '.join(detected_skills) if detected_skills else 'None'}")
    print(f"Priority: {raw_task_dict.get('Task_Priority', 'Low')}")
    print(f"Estimated hours: {raw_task_dict.get('Estimated_Planned_Hours', 'N/A')}")
    print()
    
    task_df = pd.DataFrame([raw_task_dict])
    inference_df = extract_task_features(task_df, candidate_df, emp_features_df)
    
    # Predict
    X = inference_df[expected_features].copy()
    inference_df['prediction_score'] = model.predict_proba(X)[:, 1]
    
    # Sort and take top K
    top = inference_df.nlargest(top_k, 'prediction_score')
    
    results = top[['Employee_ID', 'Employee_Name', 'prediction_score', 'employee_historical_task_count']].copy()
    results['Rank'] = range(1, top_k + 1)
    results['Match_Confidence'] = (results['prediction_score'] * 100).round(1).astype(str) + '%'
    
    return results[['Rank', 'Employee_ID', 'Employee_Name', 'Match_Confidence', 'employee_historical_task_count']]


In [22]:
# Example task using Odoo vocabulary (matches training data)
new_task = {
    "Task_ID": "TSK-6099",
    "Task_Title": "Odoo Module Development for Inventory",
    "Task_Description": "Develop custom Odoo addon for inventory management with database integration and server deployment.",
    "Task_Priority": "Normal",
    "Estimated_Planned_Hours": 24.0,
    "Days_To_Deadline": 7
}

recommendations = recommend_employees(
    raw_task_dict=new_task,
    candidate_df=candidate_pool,
    model=xgb_model,
    emp_features_df=emp_features,
    expected_features=model_feature_columns,
    top_k=5
)

print("TOP EMPLOYEE RECOMMENDATIONS")
print("=" * 75)
print(recommendations.to_string(index=False))
print("=" * 75)



Analyzing Task: Odoo Module Development for Inventory...
Detected skills: Odoo_ERP_Development, Database_Management, Server_Administration, Web_Development
Priority: Normal
Estimated hours: 24.0



TOP EMPLOYEE RECOMMENDATIONS
 Rank Employee_ID Employee_Name Match_Confidence  employee_historical_task_count
    1       EMP-9         EMP-9            60.5%                              50
    2      EMP-41        EMP-41            37.8%                             106
    3      EMP-12        EMP-12            37.4%                              62
    4      EMP-42        EMP-42            27.2%                              67
    5      EMP-17        EMP-17            26.0%                              68


In [23]:
# Second example: DevOps task
devops_task = {
"Task_ID": "TSK-1156",
    "Project_ID": "PRJ-202",
    "Task_Title": "Perform Security Vulnerability Audit",
    "Task_Description": "Conduct a comprehensive security audit of the user authentication service to identify and patch potential SQL injection and XSS vulnerabilities.",
    "Required_Skills": "Cybersecurity, Penetration Testing, Python, OWASP",
    "Estimated_Planned_Hours": 28.5,
    "Days_To_Deadline": 12
}

devops_recs = recommend_employees(
    raw_task_dict=devops_task,
    candidate_df=candidate_pool,
    model=xgb_model,
    emp_features_df=emp_features,
    expected_features=model_feature_columns,
    top_k=5
)

print("TOP EMPLOYEE RECOMMENDATIONS (DevOps Task)")
print("=" * 75)
print(devops_recs.to_string(index=False))
print("=" * 75)



Analyzing Task: Perform Security Vulnerability Audit...
Detected skills: Database_Management
Priority: Low
Estimated hours: 28.5



TOP EMPLOYEE RECOMMENDATIONS (DevOps Task)
 Rank Employee_ID Employee_Name Match_Confidence  employee_historical_task_count
    1       EMP-9         EMP-9            36.6%                              50
    2      EMP-12        EMP-12            18.1%                              62
    3      EMP-42        EMP-42            15.7%                              67
    4      EMP-46        EMP-46            12.7%                              57
    5      EMP-27        EMP-27            11.3%                              88


In [24]:
# Display model performance metadata
metadata_path = f"{DIRS['models']}/optuna_tuning_metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        meta = json.load(f)
    
    print("=" * 70)
    print("MODEL PERFORMANCE (Test Set)")
    print("=" * 70)
    print(f"Tuned NDCG@5:     {meta.get('xgb_test_ndcg5', 0):.4f}")
    print(f"Baseline NDCG@5:  {meta.get('xgb_test_ndcg5', 0) - meta.get('xgb_test_improvement_over_baseline', 0):.4f}")
    print(f"Improvement:      {meta.get('xgb_test_improvement_over_baseline', 0)*100:+.2f}%")
    print(f"Test set status:  {meta.get('test_set_status', 'N/A')}")
    print("=" * 70)
else:
    print("No tuning metadata available")


MODEL PERFORMANCE (Test Set)
Tuned NDCG@5:     0.8443
Baseline NDCG@5:  0.8167
Improvement:      +2.76%
Test set status:  UNTOUCHED during optimization
